In [1]:
import pickle

with open("results_cv_all.pkl", "rb") as f:
    results_cv = pickle.load(f)

print("Resultados cargados correctamente")
print(type(results_cv))

Resultados cargados correctamente
<class 'dict'>


# Analisis de Varianza

In [3]:
import pandas as pd

# Lista para almacenar filas
rows = []

for model_name, model_data in results_cv.items():
    for key, metrics in model_data.items():
        # Divide solo en el primer guion bajo
        pretratamiento, rango = key.split('_', 1)
        
        rmse_list = metrics['rmse']
        r2_list = metrics['r2']
        mae_list = metrics['mae']
        rpd_list = metrics['rpd']
        
        for i in range(len(rmse_list)):
            rows.append({
                'modelo': model_name,
                'pretratamiento': pretratamiento,
                'rango': rango,
                'fold': i+1,
                'rmse': rmse_list[i],
                'r2': r2_list[i],
                'mae': mae_list[i],
                'rpd': rpd_list[i]
            })

df = pd.DataFrame(rows)

print("DataFrame construido:")
print(df.head())
print(f"\nTotal de registros: {len(df)}")

DataFrame construido:
  modelo pretratamiento     rango  fold      rmse        r2       mae  \
0    svr           orig  completo     1  4.551582  0.950369  3.383833   
1    svr           orig  completo     2  4.468273  0.949654  3.295180   
2    svr           orig  completo     3  4.365723  0.950988  3.246745   
3    svr           orig  completo     4  4.372491  0.956107  3.248756   
4    svr           orig  completo     5  4.230384  0.953597  3.126991   

        rpd  
0  4.490303  
1  4.458295  
2  4.518576  
3  4.774794  
4  4.643833  

Total de registros: 180


In [5]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

modelo_anova = ols(
    'rmse ~ C(modelo) + C(pretratamiento) + C(rango) + '
    'C(modelo):C(pretratamiento) + C(modelo):C(rango) + C(pretratamiento):C(rango)',
    data=df
).fit()

anova_table = sm.stats.anova_lm(modelo_anova, typ=2)
print(anova_table)

                                sum_sq     df           F        PR(>F)
C(modelo)                    75.038394    2.0  667.350211  3.461541e-77
C(pretratamiento)             8.372338    3.0   49.639311  1.397212e-22
C(rango)                      4.495884    2.0   39.983917  9.576347e-15
C(modelo):C(pretratamiento)   9.686746    6.0   28.716197  5.553186e-23
C(modelo):C(rango)           71.527675    4.0  318.063904  6.869489e-74
C(pretratamiento):C(rango)    1.496098    6.0    4.435158  3.593953e-04
Residual                      8.770500  156.0         NaN           NaN


# Post‑hoc con Tukey

In [7]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd
for modelo in df['modelo'].unique():
    subset = df[df['modelo'] == modelo]
    
    tukey = pairwise_tukeyhsd(
        endog=subset['rmse'],
        groups=subset['rango'],
        alpha=0.05
    )
    
    print(f"\nTukey para modelo = {modelo} (comparando rangos)")
    print(tukey)


Tukey para modelo = svr (comparando rangos)
    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
  group1     group2   meandiff p-adj   lower   upper  reject
------------------------------------------------------------
  completo hasta_1400   2.2723    0.0  2.0591  2.4855   True
  completo hasta_1700   0.2674 0.0104  0.0542  0.4806   True
hasta_1400 hasta_1700  -2.0049    0.0 -2.2181 -1.7917   True
------------------------------------------------------------

Tukey para modelo = random_forest (comparando rangos)
    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
  group1     group2   meandiff p-adj   lower   upper  reject
------------------------------------------------------------
  completo hasta_1400  -0.8102    0.0 -1.1733  -0.447   True
  completo hasta_1700   -0.374 0.0422 -0.7371 -0.0108   True
hasta_1400 hasta_1700   0.4362 0.0148   0.073  0.7993   True
------------------------------------------------------------

Tukey para modelo = cnn (comparando rangos)
 

In [8]:
for modelo in df['modelo'].unique():
    for rango in df['rango'].unique():
        subset = df[
            (df['modelo'] == modelo) &
            (df['rango'] == rango)
        ]
        
        tukey = pairwise_tukeyhsd(
            endog=subset['rmse'],
            groups=subset['pretratamiento'],
            alpha=0.05
        )
        
        print(f"\nTukey para modelo={modelo}, rango={rango}")
        print(tukey)


Tukey para modelo=svr, rango=completo
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
   msc   orig  -0.0381  0.964 -0.2689  0.1927  False
   msc     sg   -0.645    0.0 -0.8758 -0.4142   True
   msc    snv  -0.0629 0.8624 -0.2937  0.1679  False
  orig     sg  -0.6069    0.0 -0.8377 -0.3761   True
  orig    snv  -0.0248 0.9895 -0.2556   0.206  False
    sg    snv   0.5821    0.0  0.3513  0.8129   True
----------------------------------------------------

Tukey para modelo=svr, rango=hasta_1700
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
   msc   orig    0.282 0.0139  0.0522  0.5119   True
   msc     sg  -0.3009 0.0086 -0.5308  -0.071   True
   msc    snv  -0.1431 0.3178 -0.3729  0.0868  False
  orig     sg  -0.5829    0.0 -0.8128 -0.3531   True
  orig    snv  -0.4

In [9]:
for modelo in df['modelo'].unique():
    for pre in df['pretratamiento'].unique():
        subset = df[
            (df['modelo'] == modelo) &
            (df['pretratamiento'] == pre)
        ]
        
        tukey = pairwise_tukeyhsd(
            endog=subset['rmse'],
            groups=subset['rango'],
            alpha=0.05
        )
        
        print(f"\nTukey para modelo={modelo}, pretratamiento={pre}")
        print(tukey)


Tukey para modelo=svr, pretratamiento=orig
    Multiple Comparison of Means - Tukey HSD, FWER=0.05     
  group1     group2   meandiff p-adj   lower   upper  reject
------------------------------------------------------------
  completo hasta_1400   2.3692    0.0  2.1002  2.6381   True
  completo hasta_1700   0.4415 0.0024  0.1726  0.7105   True
hasta_1400 hasta_1700  -1.9276    0.0 -2.1965 -1.6587   True
------------------------------------------------------------

Tukey para modelo=svr, pretratamiento=sg
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
  group1     group2   meandiff p-adj   lower  upper  reject
-----------------------------------------------------------
  completo hasta_1400   2.4163    0.0  2.1784 2.6543   True
  completo hasta_1700   0.4655 0.0006  0.2275 0.7034   True
hasta_1400 hasta_1700  -1.9509    0.0 -2.1888 -1.713   True
-----------------------------------------------------------

Tukey para modelo=svr, pretratamiento=msc
    Multiple Comparison 

# GRAFICO REAL VS PREDICHO

In [10]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from math import sqrt

def plot_predictions(y_true, y_pred, title="Real vs Predicho", 
                     xlabel="Real", ylabel="Predicción", 
                     show_metrics=True, show_regression_line=True,
                     save_path=None, 
                     figsize=(7,6), dpi=150, color='blue', alpha=0.6):
    """
    Genera un gráfico de dispersión de valores reales vs predichos
    con línea de 45° (identidad) y, opcionalmente, la línea de regresión lineal.
    """
    plt.figure(figsize=figsize)
    
    # Límites para los ejes (basados en los datos)
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    
    # Dibujar puntos
    plt.scatter(y_true, y_pred, color=color, alpha=alpha, edgecolors='k', linewidth=0.5)
    
    # Línea de identidad (45°)
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Identidad (45°)')
    
    # Línea de regresión lineal (ajuste de mínimos cuadrados)
    if show_regression_line:
        # Ajuste lineal: y_pred = m * y_true + b
        m, b = np.polyfit(y_true, y_pred, 1)
        x_line = np.linspace(min_val, max_val, 100)
        y_line = m * x_line + b
        plt.plot(x_line, y_line, 'g-', lw=2, label=f'Ajuste lineal')
    
    # Mostrar métricas (R², RMSE, MAE) en el gráfico
    if show_metrics:
        r2 = r2_score(y_true, y_pred)
        rmse = sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        textstr = f'$R^2 = {r2:.3f}$\nRMSE = {rmse:.3f}\nMAE = {mae:.3f}'
        plt.text(0.05, 0.95, textstr, transform=plt.gca().transAxes,
                 fontsize=12, verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Etiquetas y título
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    
    # Guardar si se indica
    if save_path:
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Gráfica guardada en {save_path}")
    
    plt.show()

In [11]:
def generate_parity_plots_from_cv_results(results_cv_model, model_name, output_dir="figures_cv"):
    os.makedirs(output_dir, exist_ok=True)

    for dataset, data in results_cv_model.items():
        if "y_true_oof" in data and "y_pred_oof" in data:
            y_true = np.array(data["y_true_oof"])
            y_pred = np.array(data["y_pred_oof"])

            plot_predictions(
                y_true=y_true,
                y_pred=y_pred,
                title=f"{model_name} - {dataset} (OOF, 5-fold CV)",
                save_path=f"{output_dir}/{model_name.lower()}_{dataset}_parity_oof.png"
            )
        else:
            print(f"Advertencia: No se encontraron predicciones OOF para {dataset} en {model_name}")

In [ ]:
generate_parity_plots_from_cv_results(results_cv['svr'], "SVR")
generate_parity_plots_from_cv_results(results_cv['random_forest'], "RandomForest")
generate_parity_plots_from_cv_results(results_cv['cnn'], "CNN")